# FireEye — YOLOv8m Training

Trains a YOLOv8m model on the merged wildfire dataset.

**Run time**: ~8–12 hrs on T4 · ~3–4 hrs on A100 (100 epochs)

**Prerequisites**: Run `01_dataset_prep.ipynb` first, or ensure `data/merged/` exists in Drive.

In [ ]:
# ── Cell 1: GPU check ────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q ultralytics>=8.3.0 wandb huggingface_hub
import ultralytics
ultralytics.checks()

In [ ]:
# ── Cell 3: Mount Drive + set paths ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_ROOT    = '/content/drive/MyDrive/fireye'
DATASET_ROOT = f'{REPO_ROOT}/data/merged'
RUNS_ROOT    = f'{REPO_ROOT}/runs'
os.chdir(REPO_ROOT)
print(f'Dataset: {DATASET_ROOT}')
print(f'Runs: {RUNS_ROOT}')

In [ ]:
# ── Cell 4: Patch dataset.yaml with runtime path ─────────────────────────────
import yaml, shutil

src_yaml = f'{REPO_ROOT}/configs/dataset.yaml'
runtime_yaml = '/content/dataset.yaml'

with open(src_yaml) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = DATASET_ROOT

with open(runtime_yaml, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('dataset.yaml patched:')
!cat /content/dataset.yaml

In [ ]:
# ── Cell 5: (Optional) W&B experiment tracking ───────────────────────────────
# Skip this cell if you don't want W&B logging
import wandb
wandb.login()
# The YOLO trainer auto-integrates with W&B when wandb is installed

In [ ]:
# ── Cell 6: Configure batch size for GPU ─────────────────────────────────────
import torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb >= 40:      # A100
    BATCH = 48
elif vram_gb >= 15:    # T4
    BATCH = 16
else:
    BATCH = 8

print(f'VRAM: {vram_gb:.1f} GB → batch size: {BATCH}')

In [ ]:
# ── Cell 7: TRAIN ────────────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO('yolov8m.pt')  # download COCO pretrained weights

results = model.train(
    data='/content/dataset.yaml',
    cfg=f'{REPO_ROOT}/configs/hyp_firetune.yaml',
    epochs=100,
    imgsz=640,
    batch=BATCH,
    workers=4,
    device=0,
    optimizer='AdamW',
    cos_lr=True,
    patience=20,
    amp=True,
    pretrained=True,
    save=True,
    save_period=10,
    project=RUNS_ROOT,
    name='fireye_yolov8m',
    exist_ok=True,
    verbose=True,
)

In [ ]:
# ── Cell 8: Training curves ───────────────────────────────────────────────────
from IPython.display import Image as IPImage
IPImage(f'{RUNS_ROOT}/fireye_yolov8m/results.png', width=900)

In [ ]:
# ── Cell 9: Validate on test split ───────────────────────────────────────────
best_weights = f'{RUNS_ROOT}/fireye_yolov8m/weights/best.pt'
model = YOLO(best_weights)

metrics = model.val(
    data='/content/dataset.yaml',
    split='test',
    conf=0.25,
    iou=0.45,
    device=0,
)

print(f'mAP@50      : {metrics.box.map50:.4f}  (target > 0.88)')
print(f'mAP@50-95   : {metrics.box.map:.4f}')
print(f'Precision   : {metrics.box.mp:.4f}')
print(f'Recall      : {metrics.box.mr:.4f}  (target > 0.90)')

In [ ]:
# ── Cell 10: Visual sanity check ─────────────────────────────────────────────
import glob, random
from IPython.display import Image as IPImage

sample_imgs = random.sample(
    glob.glob(f'{DATASET_ROOT}/images/test/*.jpg'), 4
)

results = model.predict(
    source=sample_imgs,
    conf=0.25,
    iou=0.45,
    save=True,
    project='/content/predictions',
    name='sanity',
    exist_ok=True,
)

for r in results:
    IPImage(r.save_dir / r.path.name, width=600)

## Done!

Best weights saved at:
`runs/fireye_yolov8m/weights/best.pt`

Open `03_export_evaluate.ipynb` to export to ONNX and push to HuggingFace.